# 02 - Model Validation Against Source Excel

This notebook compares the Python feasibility model against selected benchmark outputs from the original Provider Excel workbook.

The purpose is reconciliation, not blind replication. The v0 comparison identifies hidden or ambiguous Excel assumptions; the v1 comparison makes those assumptions explicit.


## Validation Principle

- Match source inputs where possible.
- Compare core outputs metric by metric.
- Mark differences as formula mismatch, assumption mismatch, or expected difference.
- Avoid forcing the Python model to copy unclear Excel behavior.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.financial_model import FeasibilityInputs, MachineSpec, load_machine_specs, summarize_feasibility


## Load Reconciliation Tables


In [ ]:
reconciliation_v0 = pd.read_csv(ROOT / 'data' / 'validation' / 'reconciliation_summary.csv')
reconciliation_v1 = pd.read_csv(ROOT / 'data' / 'validation' / 'reconciliation_v1_summary.csv')

reconciliation_v1


## Re-run Python v1 Benchmark Cases

The code below uses the same source inputs as the Excel dashboard and adds explicit assumptions for electricity load factor, fixed maintenance, labor, tax, and financing basis.


In [ ]:
machine_specs = load_machine_specs(ROOT / 'data' / 'processed' / 'machine_specs.csv')

def machine_by_capacity(capacity_ton: float) -> MachineSpec:
    row = machine_specs.loc[machine_specs['capacity_ton'].eq(capacity_ton)].iloc[0]
    return MachineSpec(
        model=str(row['model']),
        capacity_ton=float(row['capacity_ton']),
        total_load_kw=float(row['total_load_kw']),
        operation_hours_per_day=float(row['operation_hours_per_day']),
        asia_price_no_ce_usd=float(row['asia_price_no_ce_usd']),
        monthly_rental_ratio=float(row['monthly_rental_ratio']),
    )

capex_inputs = FeasibilityInputs(
    business_model='capex',
    waste_tons_per_day=100,
    collection_rate=1,
    days_per_month=30,
    conversion_rate=0.3,
    fertilizer_price_usd_per_ton=300,
    service_fee_usd_per_ton=50,
    electricity_rate_usd_per_kwh=0.2033,
    electricity_load_factor=0.31972454500737824,
    enzyme_cost_usd_per_ton_waste=15,
    direct_labor_cost_usd_per_month=2800,
    indirect_labor_cost_usd_per_month=18800,
    maintenance_cost_usd_per_month=4000,
    machine_discount_rate=0.55,
    grant_rate=0.10,
    down_payment_rate=0.10,
    loan_interest_expense_usd_per_month=73718.8681329931,
    initial_investment_basis='excel_financed_principal',
    roi_denominator_basis='discounted_machine_capex',
    cashflow_months=(4, 12, 12, 12, 12, 12, 12, 12, 12, 12, 8),
)
capex_result = summarize_feasibility(capex_inputs, machine_by_capacity(100))

opex_inputs = FeasibilityInputs(
    business_model='opex',
    waste_tons_per_day=100,
    collection_rate=1,
    days_per_month=30,
    conversion_rate=0.7,
    fertilizer_price_usd_per_ton=250,
    service_fee_usd_per_ton=0,
    electricity_rate_usd_per_kwh=0.2033,
    electricity_load_factor=0.34431874077717656,
    enzyme_cost_usd_per_ton_waste=0,
    direct_labor_cost_usd_per_month=0,
    maintenance_cost_usd_per_month=0,
    rental_discount_rate=0,
    tax_rate=0.10,
    cashflow_months=(4, 12, 12, 12, 12, 12, 12, 12, 12, 12, 8),
)
opex_result = summarize_feasibility(opex_inputs, machine_by_capacity(60))

pd.DataFrame([capex_result, opex_result], index=['CapEx Python v1', 'OpEx Python v1']).T


## Key Findings

1. Revenue and fertilizer output reconcile well in both v0 and v1.
2. Electricity cost reconciles only after adding an explicit load factor.
3. Maintenance cost reconciles only after allowing fixed monthly maintenance assumptions.
4. CapEx return metrics reconcile only after making initial investment and ROI denominator basis explicit.
5. OpEx ROI and IRR remain intentionally guarded because zero initial investment makes those metrics invalid.
